In [1]:
%pip install torch torchvision matplotlib ipykernel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torchvision
import torchvision.transforms as transforms

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),   # テンソルに変換（多次元配列に）
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # 正規化（データを扱いやすい範囲に）
])

# データのダウンロードと前処理の適用
# trainset：訓練用データセット（CIFAR10の画像をダウンロードする）
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
# trainloader：ダウンロードした画像から64枚ずつに分けて渡す
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=0)

# testset：テスト用データセット
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=0)

In [4]:
print(f"訓練データ（練習用）: {len(trainset)} 枚")
print(f"テストデータ（本番用）: {len(testset)} 枚")

訓練データ（練習用）: 50000 枚
テストデータ（本番用）: 10000 枚


In [5]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # 1層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv1 = nn.Conv2d(3, 6, 5)
        # プーリング層：（ウィンドウサイズ, ストライド）
        self.pool = nn.MaxPool2d(2, 2)
        # 2層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv2 = nn.Conv2d(6, 16, 5)
        
        # Affine層
        # 400本（前の層を通り抜けた後のデータの総数）を120にまとめる
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        # 120個を受け取り、84個に、
        self.fc2 = nn.Linear(120, 84)
        # 84個受取り、最終的な出口は10個になる
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # 入力データxにフィルターを掛ける➡データの形：[6, 28, 28]
        # プーリング層を通して[6, 14, 14]になる
        x = self.pool(F.relu(self.conv1(x)))
        # 2層目の畳み込み層に、さっきの[6, 14, 14]が入ってくる
        # フィルターを掛ける➡データの形：[16, 10, 10]
        # プーリング層を通して[16, 5, 5]になる
        x = self.pool(F.relu(self.conv2(x)))

        # 今のxは[16, 5, 5]の3次元 ➡ これを全結合層に入れるために1次元に変形する（全結合層は1次元の入力を受け取る層）
        # view：Numpyのreshapeと同じ
        x = x.view(x.size(0), -1)

        # 全結合層
        # [400]を受け取って、[120]に絞り込む
        x = F.relu(self.fc1(x))
        # [120]を受け取って、[84]に絞り込む
        x = F.relu(self.fc2(x))

        # [84]を受け取り、最終的な出口[10]に絞り込む
        x = self.fc3(x)
        return x

net = Net()

In [6]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

In [7]:
import time

epochs = 10

# 学習が始まる時間と終わる時間を記録して、合計何秒かかったかを計る
start_time = time.time()

for epoch in range(epochs):
    running_loss = 0.0

    # trainloader：画像を64枚ずつに小分けした箱
    for i, data in enumerate(trainloader):
        inputs, labels = data

        # 前回の問題で計算した「ズレの記録」をリセット
        optimizer.zero_grad()

        # AIモデルに予測を出させる
        outputs = net(inputs)

        # AIの予測（outputs）と本当の正解（labeks）を比べて、どれくらい間違えているかのlossを計算
        loss = criterion(outputs, labels)

        # 逆伝播で勾配を計算
        loss.backward()

        # 計算した結果をもとに、パラメータを更新
        optimizer.step()

        running_loss += loss.item()
        if i % 2000 == 1999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

end_time = time.time()

In [8]:
correct = 0
total = 0

# 評価モードにする（学習はしない）
net.eval()

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'10,000枚のテスト画像に対する正解率: {100 * correct / total:.2f} %')


10,000枚のテスト画像に対する正解率: 62.71 %


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

# Heの初期値を取り入れてみる
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # 1層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv1 = nn.Conv2d(3, 6, 5)
        # プーリング層：（ウィンドウサイズ, ストライド）
        self.pool = nn.MaxPool2d(2, 2)
        # 2層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv2 = nn.Conv2d(6, 16, 5)
        
        # Affine層
        # 400本（前の層を通り抜けた後のデータの総数）を120にまとめる
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        # 120個を受け取り、84個に、
        self.fc2 = nn.Linear(120, 84)
        # 84個受取り、最終的な出口は10個になる
        self.fc3 = nn.Linear(84, 10)

        layers = [self.conv1, self.conv2, self.fc1, self.fc2, self.fc3]
        for layer in layers:
            # 重み(w)をHeの初期値（Kaiming Normal）で初期化
            init.kaiming_normal_(layer.weight, nonlinearity='relu')
            # バイアス(b)は 0 で初期化
            if layer.bias is not None:
                init.zeros_(layer.bias)

    def forward(self, x):
        # 入力データxにフィルターを掛ける➡データの形：[6, 28, 28]
        # プーリング層を通して[6, 14, 14]になる
        x = self.pool(F.relu(self.conv1(x)))
        # 2層目の畳み込み層に、さっきの[6, 14, 14]が入ってくる
        # フィルターを掛ける➡データの形：[16, 10, 10]
        # プーリング層を通して[16, 5, 5]になる
        x = self.pool(F.relu(self.conv2(x)))

        # 今のxは[16, 5, 5]の3次元 ➡ これを全結合層に入れるために1次元に変形する（全結合層は1次元の入力を受け取る層）
        # view：Numpyのreshapeと同じ
        x = x.view(x.size(0), -1)

        # 全結合層
        # [400]を受け取って、[120]に絞り込む
        x = F.relu(self.fc1(x))
        # [120]を受け取って、[84]に絞り込む
        x = F.relu(self.fc2(x))

        # [84]を受け取り、最終的な出口[10]に絞り込む
        x = self.fc3(x)
        return x

    net = Net()

In [10]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

In [11]:
import time

epochs = 10

# 学習が始まる時間と終わる時間を記録して、合計何秒かかったかを計る
start_time = time.time()

for epoch in range(epochs):
    running_loss = 0.0

    # trainloader：画像を64枚ずつに小分けした箱
    for i, data in enumerate(trainloader):
        inputs, labels = data

        # 前回の問題で計算した「ズレの記録」をリセット
        optimizer.zero_grad()

        # AIモデルに予測を出させる
        outputs = net(inputs)

        # AIの予測（outputs）と本当の正解（labeks）を比べて、どれくらい間違えているかのlossを計算
        loss = criterion(outputs, labels)

        # 逆伝播で勾配を計算
        loss.backward()

        # 計算した結果をもとに、パラメータを更新
        optimizer.step()

        running_loss += loss.item()
        if i % 2000 == 1999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

end_time = time.time()

In [12]:
correct = 0
total = 0

# 評価モードにする（学習はしない）
net.eval()

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'10,000枚のテスト画像に対する正解率: {100 * correct / total:.2f} %')


10,000枚のテスト画像に対する正解率: 62.99 %


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

# バッチ正規化を取り入れる
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # 1層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv1 = nn.Conv2d(3, 6, 5)
        # バッチ正規化を取り入れる
        self.bn1 = nn.BatchNorm2d(6) 
        # プーリング層：（ウィンドウサイズ, ストライド）
        self.pool = nn.MaxPool2d(2, 2)
        # 2層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv2 = nn.Conv2d(6, 16, 5)
        # 2層目の畳み込み層のバッチ正規化
        self.bn2 = nn.BatchNorm2d(6) 
        
        # Affine層
        # 400本（前の層を通り抜けた後のデータの総数）を120にまとめる
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        # 3層目の全結合層のバッチ正規化
        self.bn3 = nn.BatchNorm2d(6) 
        # 120個を受け取り、84個に、
        self.fc2 = nn.Linear(120, 84)
        # 4層目のバッチ正規化
        self.bn4 = nn.BatchNorm2d(6) 
        # 84個受取り、最終的な出口は10個になる
        self.fc3 = nn.Linear(84, 10)

        layers = [self.conv1, self.conv2, self.fc1, self.fc2, self.fc3]
        for layer in layers:
            # 重み(w)をHeの初期値（Kaiming Normal）で初期化
            init.kaiming_normal_(layer.weight, nonlinearity='relu')
            # バイアス(b)は 0 で初期化
            if layer.bias is not None:
                init.zeros_(layer.bias)

    def forward(self, x):
        # 入力データxにフィルターを掛ける➡データの形：[6, 28, 28]
        # プーリング層を通して[6, 14, 14]になる
        x = self.pool(F.relu(self.conv1(x)))
        # 2層目の畳み込み層に、さっきの[6, 14, 14]が入ってくる
        # フィルターを掛ける➡データの形：[16, 10, 10]
        # プーリング層を通して[16, 5, 5]になる
        x = self.pool(F.relu(self.conv2(x)))

        # 今のxは[16, 5, 5]の3次元 ➡ これを全結合層に入れるために1次元に変形する（全結合層は1次元の入力を受け取る層）
        # view：Numpyのreshapeと同じ
        x = x.view(x.size(0), -1)

        # 全結合層
        # [400]を受け取って、[120]に絞り込む
        x = F.relu(self.fc1(x))
        # [120]を受け取って、[84]に絞り込む
        x = F.relu(self.fc2(x))

        # [84]を受け取り、最終的な出口[10]に絞り込む
        x = self.fc3(x)
        return x

    net = Net()

In [14]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.0003)

In [15]:
import time

epochs = 10

# 学習が始まる時間と終わる時間を記録して、合計何秒かかったかを計る
start_time = time.time()

for epoch in range(epochs):
    running_loss = 0.0

    # trainloader：画像を64枚ずつに小分けした箱
    for i, data in enumerate(trainloader):
        inputs, labels = data

        # 前回の問題で計算した「ズレの記録」をリセット
        optimizer.zero_grad()

        # AIモデルに予測を出させる
        outputs = net(inputs)

        # AIの予測（outputs）と本当の正解（labeks）を比べて、どれくらい間違えているかのlossを計算
        loss = criterion(outputs, labels)

        # 逆伝播で勾配を計算
        loss.backward()

        # 計算した結果をもとに、パラメータを更新
        optimizer.step()

        running_loss += loss.item()
        if i % 2000 == 1999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

end_time = time.time()

In [16]:
correct = 0
total = 0

# 評価モードにする（学習はしない）
net.eval()

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'10,000枚のテスト画像に対する正解率: {100 * correct / total:.2f} %')


10,000枚のテスト画像に対する正解率: 63.24 %


In [17]:

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'10,000枚のテスト画像に対する正解率: {100 * correct / total:.2f} %')


10,000枚のテスト画像に対する正解率: 63.24 %


In [22]:
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

# ネットワークを広げてみる
# エポック数を20に換える
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # 1層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv1 = nn.Conv2d(3, 32, 5)
        # バッチ正規化を取り入れる
        self.bn1 = nn.BatchNorm2d(32) 
        # プーリング層：（ウィンドウサイズ, ストライド）
        self.pool = nn.MaxPool2d(2, 2)
        # 2層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        # 2層目の畳み込み層のバッチ正規化
        self.bn2 = nn.BatchNorm2d(64) 
        
        # Affine層
        # 400本（前の層を通り抜けた後のデータの総数）を120にまとめる
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        # 3層目の全結合層のバッチ正規化
        self.bn3 = nn.BatchNorm2d(256) 
        # 120個を受け取り、84個に、
        self.fc2 = nn.Linear(256, 84)
        # 4層目のバッチ正規化
        self.bn4 = nn.BatchNorm2d(84) 
        # 84個受取り、最終的な出口は10個になる
        self.fc3 = nn.Linear(84, 10)

        layers = [self.conv1, self.conv2, self.fc1, self.fc2, self.fc3]
        for layer in layers:
            # 重み(w)をHeの初期値（Kaiming Normal）で初期化
            init.kaiming_normal_(layer.weight, nonlinearity='relu')
            # バイアス(b)は 0 で初期化
            if layer.bias is not None:
                init.zeros_(layer.bias)

    def forward(self, x):
        # 入力データxにフィルターを掛ける➡データの形：[6, 28, 28]
        # プーリング層を通して[6, 14, 14]になる
        x = self.pool(F.relu(self.conv1(x)))
        # 2層目の畳み込み層に、さっきの[6, 14, 14]が入ってくる
        # フィルターを掛ける➡データの形：[16, 10, 10]
        # プーリング層を通して[16, 5, 5]になる
        x = self.pool(F.relu(self.conv2(x)))

        # 今のxは[16, 5, 5]の3次元 ➡ これを全結合層に入れるために1次元に変形する（全結合層は1次元の入力を受け取る層）
        # view：Numpyのreshapeと同じ
        x = x.view(x.size(0), -1)

        # 全結合層
        # [400]を受け取って、[120]に絞り込む
        x = F.relu(self.fc1(x))
        # [120]を受け取って、[84]に絞り込む
        x = F.relu(self.fc2(x))

        # [84]を受け取り、最終的な出口[10]に絞り込む
        x = self.fc3(x)
        return x

    net = Net()

In [23]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

In [24]:
import time

epochs = 20

# 学習が始まる時間と終わる時間を記録して、合計何秒かかったかを計る
start_time = time.time()

for epoch in range(epochs):
    running_loss = 0.0

    # trainloader：画像を64枚ずつに小分けした箱
    for i, data in enumerate(trainloader):
        inputs, labels = data

        # 前回の問題で計算した「ズレの記録」をリセット
        optimizer.zero_grad()

        # AIモデルに予測を出させる
        outputs = net(inputs)

        # AIの予測（outputs）と本当の正解（labeks）を比べて、どれくらい間違えているかのlossを計算
        loss = criterion(outputs, labels)

        # 逆伝播で勾配を計算
        loss.backward()

        # 計算した結果をもとに、パラメータを更新
        optimizer.step()

        running_loss += loss.item()
        if i % 2000 == 1999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

end_time = time.time()

In [25]:
correct = 0
total = 0

# 評価モードにする（学習はしない）
net.eval()

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'10,000枚のテスト画像に対する正解率: {100 * correct / total:.2f} %')


10,000枚のテスト画像に対する正解率: 59.58 %


In [26]:
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

# 過学習を防ぐDropoutを導入
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # 1層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv1 = nn.Conv2d(3, 32, 5)
        # バッチ正規化を取り入れる
        self.bn1 = nn.BatchNorm2d(32) 
        # プーリング層：（ウィンドウサイズ, ストライド）
        self.pool = nn.MaxPool2d(2, 2)
        # 2層目の畳み込み層：（入力チャンネル数, フィルター数, フィルターサイズ）
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        # 2層目の畳み込み層のバッチ正規化
        self.bn2 = nn.BatchNorm2d(64) 
        
        # Affine層
        # 400本（前の層を通り抜けた後のデータの総数）を120にまとめる
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        # 3層目の全結合層のバッチ正規化
        self.bn3 = nn.BatchNorm2d(256) 
        # 120個を受け取り、84個に、
        self.fc2 = nn.Linear(256, 84)
        # 4層目のバッチ正規化
        self.bn4 = nn.BatchNorm2d(84) 
        # 84個受取り、最終的な出口は10個になる
        self.fc3 = nn.Linear(84, 10)

        self.dropout = nn.Dropout(0.5)

        layers = [self.conv1, self.conv2, self.fc1, self.fc2, self.fc3]
        for layer in layers:
            # 重み(w)をHeの初期値（Kaiming Normal）で初期化
            init.kaiming_normal_(layer.weight, nonlinearity='relu')
            # バイアス(b)は 0 で初期化
            if layer.bias is not None:
                init.zeros_(layer.bias)

    def forward(self, x):
        # 入力データxにフィルターを掛ける➡データの形：[6, 28, 28]
        # プーリング層を通して[6, 14, 14]になる
        x = self.pool(F.relu(self.conv1(x)))
        # 2層目の畳み込み層に、さっきの[6, 14, 14]が入ってくる
        # フィルターを掛ける➡データの形：[16, 10, 10]
        # プーリング層を通して[16, 5, 5]になる
        x = self.pool(F.relu(self.conv2(x)))

        # 今のxは[16, 5, 5]の3次元 ➡ これを全結合層に入れるために1次元に変形する（全結合層は1次元の入力を受け取る層）
        # view：Numpyのreshapeと同じ
        x = x.view(x.size(0), -1)

        # 全結合層
        # [400]を受け取って、[120]に絞り込む
        x = F.relu(self.fc1(x))
        # Dropout
        x = self.dropout(x)
        # [120]を受け取って、[84]に絞り込む
        x = F.relu(self.fc2(x))
        # Dropout
        x = self.dropout(x)

        # [84]を受け取り、最終的な出口[10]に絞り込む
        x = self.fc3(x)
        return x

    net = Net()

In [27]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

In [28]:
import time

epochs = 20

# 学習が始まる時間と終わる時間を記録して、合計何秒かかったかを計る
start_time = time.time()

for epoch in range(epochs):
    running_loss = 0.0

    # trainloader：画像を64枚ずつに小分けした箱
    for i, data in enumerate(trainloader):
        inputs, labels = data

        # 前回の問題で計算した「ズレの記録」をリセット
        optimizer.zero_grad()

        # AIモデルに予測を出させる
        outputs = net(inputs)

        # AIの予測（outputs）と本当の正解（labeks）を比べて、どれくらい間違えているかのlossを計算
        loss = criterion(outputs, labels)

        # 逆伝播で勾配を計算
        loss.backward()

        # 計算した結果をもとに、パラメータを更新
        optimizer.step()

        running_loss += loss.item()
        if i % 2000 == 1999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

end_time = time.time()

In [29]:
correct = 0
total = 0

# 評価モードにする（学習はしない）
net.eval()

with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'10,000枚のテスト画像に対する正解率: {100 * correct / total:.2f} %')


10,000枚のテスト画像に対する正解率: 58.70 %
